# Assignment: 情感分析与词嵌入本实验使用 **电影评论数据集** 进行二分类情感分析，对比不同嵌入方法的效果。## 对比实验1. 随机初始化 Embedding2. 预训练 GloVe Embedding（冻结）3. 预训练 GloVe Embedding（微调）

In [ ]:
import torchimport torchtextimport collectionsimport osimport numpy as npfrom tqdm import tqdmdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'使用设备: {device}')

## 1. 创建示例数据集我们创建一个小型电影评论数据集进行演示。

In [ ]:
# 示例电影评论数据集reviews = [    # 正面评论    (1, 'This movie was absolutely fantastic and I loved every minute of it'),    (1, 'Great acting and wonderful storyline highly recommended'),    (1, 'One of the best films I have seen this year brilliant performances'),    (1, 'Amazing cinematography and touching story a masterpiece'),    (1, 'The director did an excellent job this film is a gem'),    (1, 'Loved the characters and the plot was very engaging'),    (1, 'A wonderful experience from start to finish'),    (1, 'The movie exceeded my expectations truly remarkable'),    (1, 'Fantastic performances by the entire cast'),    (1, 'A beautiful and heartwarming story'),    # 负面评论    (0, 'This movie was terrible and a complete waste of time'),    (0, 'Boring plot and bad acting not worth watching'),    (0, 'One of the worst films I have ever seen disappointing'),    (0, 'The story made no sense and the acting was awful'),    (0, 'I regret watching this movie it was horrible'),    (0, 'Poor direction and weak script a disaster'),    (0, 'A waste of money the movie was dull'),    (0, 'The film failed to deliver on its promises'),    (0, 'Terrible screenplay and uninspired performances'),    (0, 'A forgettable and poorly made movie'),]# 划分训练集和测试集train_data = reviews[:16]test_data = reviews[16:]print(f'训练集: {len(train_data)} 条')print(f'测试集: {len(test_data)} 条')print(f'示例: {train_data[0]}')

## 2. 构建词汇表

In [ ]:
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')# 构建词汇表counter = collections.Counter()for label, text in train_data + test_data:    counter.update(tokenizer(text))# torchtext 0.6.0 使用 Vocab 类vocab = torchtext.vocab.Vocab(counter, min_freq=1)vocab_size = len(vocab)print(f'词汇表大小: {vocab_size}')stoi = vocab.stoidef encode(text):    return [stoi.get(t, 0) for t in tokenizer(text)]# 测试编码sample = train_data[0][1]print(f'原文: {sample}')print(f'编码: {encode(sample)}')

## 3. 定义分类器模型

In [ ]:
class SentimentClassifier(torch.nn.Module):    def __init__(self, vocab_size, embed_dim, num_class, pretrained_embeddings=None, freeze=False):        super().__init__()        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)                if pretrained_embeddings is not None:            self.embedding.weight.data.copy_(pretrained_embeddings)            if freeze:                self.embedding.weight.requires_grad = False                self.fc = torch.nn.Linear(embed_dim, num_class)        def forward(self, text, offsets):        embedded = self.embedding(text, offsets)        return self.fc(embedded)

## 4. 训练与评估函数

In [ ]:
def collate_batch(batch):    labels, texts = [], []    for label, text in batch:        labels.append(label)        texts.append(torch.tensor(encode(text)))        offsets = [0] + [len(t) for t in texts]    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)        return (        torch.tensor(labels, dtype=torch.long),        torch.cat(texts),        offsets    )def train_model(model, train_data, epochs=50, lr=0.01):    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)    loss_fn = torch.nn.CrossEntropyLoss()        model.train()    for epoch in range(epochs):        total_loss, correct = 0, 0        for label, text in train_data:            labels, texts, offsets = collate_batch([(label, text)])            labels, texts, offsets = labels.to(device), texts.to(device), offsets.to(device)                        optimizer.zero_grad()            output = model(texts, offsets)            loss = loss_fn(output.unsqueeze(0), labels)            loss.backward()            optimizer.step()                        total_loss += loss.item()            correct += (output.argmax(1) == labels).sum().item()        return correct / len(train_data)def evaluate_model(model, test_data):    model.eval()    correct = 0    with torch.no_grad():        for label, text in test_data:            labels, texts, offsets = collate_batch([(label, text)])            labels, texts, offsets = labels.to(device), texts.to(device), offsets.to(device)            output = model(texts, offsets)            correct += (output.argmax(1) == labels).sum().item()    return correct / len(test_data)

## 5. 实验 1: 随机初始化 Embedding

In [ ]:
embed_dim = 50model_random = SentimentClassifier(vocab_size, embed_dim, 2).to(device)acc_random = train_model(model_random, train_data, epochs=100)test_acc_random = evaluate_model(model_random, test_data)print(f'随机初始化 Embedding:')print(f'  训练准确率: {acc_random:.2%}')print(f'  测试准确率: {test_acc_random:.2%}')

## 6. 实验 2: 加载 GloVe 预训练 Embedding

In [ ]:
# 加载 GloVe 预训练词向量glove = torchtext.vocab.GloVe(name='6B', dim=50)print(f'GloVe 词汇量: {len(glove)}')

In [ ]:
# 构建预训练嵌入矩阵pretrained_weights = torch.zeros(vocab_size, embed_dim)found_words = 0for word, idx in stoi.items():    if word in glove.stoi:        pretrained_weights[idx] = glove.vectors[glove.stoi[word]]        found_words += 1    else:        pretrained_weights[idx] = torch.randn(embed_dim) * 0.1print(f'词汇表中 {found_words}/{vocab_size} 个词在 GloVe 中找到')

### 实验 2a: GloVe Embedding（冻结，不更新）

In [ ]:
model_frozen = SentimentClassifier(vocab_size, embed_dim, 2, pretrained_weights, freeze=True).to(device)acc_frozen = train_model(model_frozen, train_data, epochs=100)test_acc_frozen = evaluate_model(model_frozen, test_data)print(f'GloVe Embedding (冻结):')print(f'  训练准确率: {acc_frozen:.2%}')print(f'  测试准确率: {test_acc_frozen:.2%}')

### 实验 2b: GloVe Embedding（微调，允许更新）

In [ ]:
model_finetune = SentimentClassifier(vocab_size, embed_dim, 2, pretrained_weights, freeze=False).to(device)acc_finetune = train_model(model_finetune, train_data, epochs=100)test_acc_finetune = evaluate_model(model_finetune, test_data)print(f'GloVe Embedding (微调):')print(f'  训练准确率: {acc_finetune:.2%}')print(f'  测试准确率: {test_acc_finetune:.2%}')

## 7. 结果对比

In [ ]:
import pandas as pdresults = pd.DataFrame({    '方法': ['随机初始化', 'GloVe (冻结)', 'GloVe (微调)'],    '训练准确率': [acc_random, acc_frozen, acc_finetune],    '测试准确率': [test_acc_random, test_acc_frozen, test_acc_finetune]})print('='*60)print('实验结果对比')print('='*60)print(results.to_string(index=False))print('='*60)

## 8. 语义相似度演示使用 GloVe 预训练向量计算词之间的相似度。

In [ ]:
def word_similarity(w1, w2, embeddings=glove):    if w1 in embeddings.stoi and w2 in embeddings.stoi:        v1 = embeddings.vectors[embeddings.stoi[w1]]        v2 = embeddings.vectors[embeddings.stoi[w2]]        sim = torch.dot(v1, v2) / (torch.norm(v1) * torch.norm(v2))        return sim.item()    return None# 情感词相似度word_pairs = [    ('good', 'great'),    ('good', 'bad'),    ('excellent', 'amazing'),    ('terrible', 'horrible'),    ('movie', 'film'),    ('love', 'hate')]print('词汇相似度 (余弦相似度):')print('-'*50)for w1, w2 in word_pairs:    sim = word_similarity(w1, w2)    print(f'{w1:12} vs {w2:12}: {sim:.4f}' if sim else f'{w1} 或 {w2} 不在词汇表中')

## 9. 词类比演示经典例子: king - man + woman = queen

In [ ]:
def find_analogy(a, b, c, embeddings=glove, top_k=5):    '''找出 d，使得 a:b :: c:d'''    if all(w in embeddings.stoi for w in [a, b, c]):        va = embeddings.vectors[embeddings.stoi[a]]        vb = embeddings.vectors[embeddings.stoi[b]]        vc = embeddings.vectors[embeddings.stoi[c]]                # d = b - a + c        vd = vb - va + vc                # 找最近的词        similarities = torch.matmul(embeddings.vectors, vd)        top_indices = similarities.topk(top_k + 3).indices.tolist()                results = []        for idx in top_indices:            word = embeddings.itos[idx]            if word not in [a, b, c]:                results.append(word)                if len(results) >= top_k:                    break        return results    return Noneanalogies = [    ('man', 'king', 'woman'),    ('bad', 'terrible', 'good'),    ('france', 'paris', 'italy'),]print('词类比实验:')print('-'*50)for a, b, c in analogies:    results = find_analogy(a, b, c)    if results:        print(f'{a} : {b} :: {c} : {results[0]}')        print(f'  其他候选: {results[1:]}')    print()

## 总结### 发现1. **预训练嵌入的优势**: GloVe 预训练嵌入包含了丰富的语义信息，可以帮助模型更好地理解词汇关系2. **冻结 vs 微调**:   - 冻结适合数据量小、防止过拟合的场景   - 微调适合数据量大、希望适应特定任务的场景3. **语义相似度**: 预训练嵌入可以捕获词之间的语义关系4. **词类比**: 经典的 king-man+woman=queen 展示了嵌入空间的几何特性### 局限性1. 示例数据集较小，结论可能不够可靠2. 未考虑词序和上下文（需要 BERT 等模型）3. 预训练嵌入可能不包含领域特定词汇